## Feature Selection Strategy Implementation

This notebook implements the feature-selection pipeline from the plan: broad univariate filtering, multicollinearity reduction, and RFECV-based ordering of the selected Stage 2 features.

### Covered Workflow
- Stage 1: Broad univariate filtering (500 -> ~80)
- Stage 2: Multicollinearity and L1 regularization (80 -> ~25)
- Stage 3: RFECV ordering of Stage 2 features for top-k model comparisons

### Included Components
- Data loading and sanity checks
- Custom business scorer aligned with project metric
- Stage-by-stage feature reduction and validation
- RFECV ranking of Stage 2 features
- Evaluation of models on multiple feature-set sizes
- Final feature locking and export artifacts

## Key Parameters

| Parameter | Default | Description |
|---|---|---|
| custom_scorer | Required | Scoring function: scorer(estimator, X, y) -> float |
| stage1_n_features | 80 | Target features after Stage 1 |
| stage2_n_features | None | Optional strict target after Stage 2 |
| stage2_n_features_band | (20, 30) | Preferred approximate Stage 2 band (data-driven final count) |
| correlation_threshold | 0.85 | Correlation pruning threshold |
| vif_threshold | 5.0 | VIF filtering threshold |
| variance_threshold | 0.01 | Variance threshold (Stage 1) |
| rfecv_step | 1 | Step size for RFECV feature elimination |
| rfecv_cv | 5 | Cross-validation folds used by RFECV |
| rfecv_n_jobs | -1 | Parallel jobs for RFECV fitting |
| top_k_feature_sets | 1..len(stage2_features) | Ranked feature subset sizes evaluated after RFECV |
| random_state | 42 | Random seed |

In [1]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.feature_selection import RFECV
from sklearn.model_selection import cross_val_score

# Add src to path
PROJECT_ROOT: Path = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

In [2]:
from cost_effective.dataset import (
    MulticollinearityFilter,
    UnivariateFeatureFilter,
    custom_scorer,
    get_classifier,
)

# Configuration
np.random.seed(42)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# Constants
DATA_PATH: Path = PROJECT_ROOT / "data"
OUTPUTS_PATH: Path = PROJECT_ROOT / "outputs"

In [3]:
# Load raw data
X_all = pd.read_csv(DATA_PATH / "x_train.txt", sep=r"\s+", header=None, low_memory=False).apply(
    pd.to_numeric, errors="coerce"
)
y_raw = pd.to_numeric(
    pd.read_csv(DATA_PATH / "y_train.txt", sep=r"\s+", header=None).iloc[:, 0], errors="coerce"
)

# Keep only rows with valid binary target and aligned feature rows.
valid_mask = y_raw.notna() & y_raw.isin([0, 1])
X_all = X_all.loc[valid_mask].reset_index(drop=True)
y = y_raw.loc[valid_mask].astype(int).reset_index(drop=True)

n_features_all = X_all.shape[1]
X_all.columns = [f"var_{i}" for i in range(n_features_all)]
y.name = "target"

print(f"✓ Loaded raw data: {X_all.shape}")
print(f"  Class distribution: {y.value_counts().to_dict()}")

✓ Loaded raw data: (5000, 500)
  Class distribution: {0: 2512, 1: 2488}


In [ ]:
# Stage 1 and Stage 2
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message=".*'penalty' was deprecated.*")
    warnings.filterwarnings("ignore", message=".*Inconsistent values: penalty=l1.*")

    print("=" * 70)
    print("STAGE 1: UNIVARIATE FILTERING (500 → 80)")
    print("=" * 70 + "\n")

    stage1_filter = UnivariateFeatureFilter(variance_threshold=0.01)
    X_stage1, stage1_features = stage1_filter.fit_transform(X_all, y, n_features=80)

    print()
    stage2_filter = MulticollinearityFilter(
        correlation_threshold=0.85,
        vif_threshold=5.0,
        random_state=42,
    )
    X_stage2, stage2_features = stage2_filter.fit_transform(
        X_stage1,
        y,
        n_features_target=None,
        n_features_band=(20, 30),
    )

print(f"✓ Stage 1 output: {len(stage1_features)} features")
print(f"✓ Stage 2 output: {len(stage2_features)} features")

STAGE 1: UNIVARIATE FILTERING (500 → 80)

[Stage 1a] Variance Threshold: 500 → 500 features
[Stage 1b] Mutual Information computed for 500 features
  Top 10 MI scores:
     feature  mi_score
10    var_10  0.029785
456  var_456  0.021367
470  var_470  0.020561
312  var_312  0.018987
159  var_159  0.018163
415  var_415  0.018163
175  var_175  0.018041
254  var_254  0.017258
4      var_4  0.017163
379  var_379  0.017061

[Stage 1c] LightGBM Importance computed
  Top 10 LGBM Importances:
     feature  lgbm_importance
214  var_214               94
379  var_379               94
341  var_341               81
254  var_254               73
190  var_190               71
116  var_116               67
159  var_159               64
226  var_226               61
389  var_389               60
482  var_482               58

[Stage 1 Output] Selected 80 features out of 500

STAGE 2: MULTICOLLINEARITY & L1 REGULARIZATION FILTERING

[Stage 2a] Correlation Pruning (threshold=0.85):
  High-correlation pair

In [ ]:
# Stage 3
print("=" * 70)
print("STAGE 3: RFECV WITH CUSTOM BUSINESS SCORER")
print("=" * 70)

# Configure RFECV with custom scorer
rfecv = RFECV(
    estimator=get_classifier(y=y),
    step=1,
    cv=5,
    scoring=custom_scorer,
    n_jobs=-1,
    verbose=0,
)

# Fit on ndarray and silence known sklearn feature-name warning from RFECV internals.
X_stage2_np = X_stage2.to_numpy()
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message=".*X does not have valid feature names.*")
    rfecv.fit(X_stage2_np, y)

print("\n✓ RFECV completed")
print(f"  Ranking vector: {rfecv.ranking_}")

STAGE 3: RFECV WITH CUSTOM BUSINESS SCORER

✓ RFECV completed
  Ranking vector: [ 3  2 22  7  1 14  5 10 16 18 13 20 15 19  4  6 11  8 21 26 23  9 12 24
 25 17]


In [6]:
df = pd.DataFrame({
    "feature": X_stage2.columns,
    "ranking": rfecv.ranking_,
})
df.to_csv(OUTPUTS_PATH / "feature_selection_results.csv", index=False)

In [7]:
for size in range(1, len(stage2_features) + 1):
    mask = rfecv.ranking_ <= size
    selected_feats = X_stage2.columns[mask].tolist()
    cv_score = cross_val_score(
        get_classifier(y=y),
        X_stage2[selected_feats],
        y,
        cv=5,
        scoring=custom_scorer,
    ).mean()
    print(f"  Top {size:2d} features: CV score = {cv_score:.2f} points")

  Top  1 features: CV score = 2154.00 points
  Top  2 features: CV score = 2012.00 points
  Top  3 features: CV score = 1863.00 points
  Top  4 features: CV score = 1593.00 points
  Top  5 features: CV score = 1423.00 points
  Top  6 features: CV score = 1300.00 points
  Top  7 features: CV score = 1074.00 points
  Top  8 features: CV score = 822.00 points
  Top  9 features: CV score = 636.00 points
  Top 10 features: CV score = 430.00 points
  Top 11 features: CV score = 231.00 points
  Top 12 features: CV score = 20.00 points
  Top 13 features: CV score = -153.00 points
  Top 14 features: CV score = -325.00 points
  Top 15 features: CV score = -600.00 points
  Top 16 features: CV score = -769.00 points
  Top 17 features: CV score = -942.00 points
  Top 18 features: CV score = -1158.00 points
  Top 19 features: CV score = -1313.00 points
  Top 20 features: CV score = -1550.00 points
  Top 21 features: CV score = -1733.00 points
  Top 22 features: CV score = -1938.00 points
  Top 23 fe